In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# scd2.employee_detail.source_employees --> source employees
# scd2.employee_detail.target_employees --> target employees
df = spark.table('scd2.employee_detail.source_employees')
target = DeltaTable.forName(spark, 'scd2.employee_detail.target_employees')

In [0]:
dedup_df = df.withColumn('row_hash', sha2(concat(*df.columns), 256))
dedup_df = dedup_df.withColumn('rnk', row_number().over(Window.partitionBy(col('emp_id')).orderBy(col('emp_id').desc(),col('salary').desc(),col('row_hash').desc())))
dedup_df = dedup_df.filter(col('rnk') == 1)

In [0]:
incoming_df = dedup_df.withColumn('start_date', current_date()) \
    .withColumn('end_date', lit(None)) \
        .withColumn('active', lit(True))

In [0]:
target.alias('t').merge(incoming_df.alias('s'), condition = ('t.emp_id = s.emp_id and t.active = True')) \
    .whenMatchedUpdate(
condition = """
      NOT (
        t.emp_name     <=> s.name AND
        t.emp_dept     <=> s.dept AND
        t.emp_position <=> s.position AND
        t.emp_salary   <=> s.salary
      )
    """,
    set = {
        "end_date": current_date(),
        "active":   lit(False)
    }
) .execute()

In [0]:
target.alias('t').merge(incoming_df.alias('s'), condition = ('t.emp_id = s.emp_id and (t.active = True or t.end_date is null)'))\
    .whenNotMatchedInsert(
        values = {
            'emp_id' : 's.emp_id',
            'emp_name' : 's.name',
            'emp_dept' : 's.dept',
            'emp_position' : 's.position',
            'emp_salary' : 's.salary',
            'start_date' : 's.start_date',
            'end_date' : 's.end_date',
            'active' : 's.active'

        }
    ).execute()